# Rigol DG5502 Pro USB discovery
Run in a local Python kernel on this laptop, with the powered generator connected to its USB DEVICE port. Only `*IDN?` is sent; output settings are unchanged.

On Windows install NI-VISA (or a compatible vendor VISA runtime) with USB support. PyVISA needs a VISA backend as well as the Python package. Run the optional installation cell if needed.
## Local measurement storage

Calibration captures, LUTs, figures and executed-notebook backups are stored
outside the API/SCPI checkout. The Windows default is
`C:/Users/kairan/RedPitaya_local_data/RedPitaya_app`; result subdirectories
retain their `scpi-tests/tdc_*_results` names. Linux defaults to
`~/.local/share/redpitaya/RedPitaya_app`. Set `REDPITAYA_DATA_DIR` to an absolute
external directory (or a VirtualBox shared host folder) before starting the
kernel to override it. See `local_data.py`. Restart the kernel after this
storage migration. Historical outputs were archived before clearing this copy.


In [ ]:
# Uncomment if needed, then restart the kernel:
# %pip install pyvisa

In [ ]:
import sys
import pyvisa

print("Python:", sys.executable)
print("PyVISA:", pyvisa.__version__)
VISA_BACKEND = ""  # Default vendor VISA. "@py" requires additional USB setup.
RESOURCE_OVERRIDE = ""  # Exact USB...::INSTR address, if multiple generators exist.
TIMEOUT_MS = 5000

In [ ]:
# Close previous sessions when rerunning.
if globals().get("generator") is not None:
    generator.close()
    generator = None
if globals().get("rm") is not None:
    rm.close()
    rm = None
try:
    rm = pyvisa.ResourceManager(VISA_BACKEND)
except (OSError, ValueError, ImportError) as exc:
    raise RuntimeError("Cannot load VISA. Install a VISA runtime with USB support and restart the kernel.") from exc
print("Backend:", rm.visalib)
resources = rm.list_resources("?*")
print("All VISA resources:", resources)
usb_resources = [r for r in resources if r.upper().startswith("USB") and r.upper().endswith("::INSTR")]
if RESOURCE_OVERRIDE:
    if not (RESOURCE_OVERRIDE.upper().startswith("USB") and RESOURCE_OVERRIDE.upper().endswith("::INSTR")):
        raise ValueError("Expected a USB instrument resource ending in ::INSTR.")
    usb_resources = [RESOURCE_OVERRIDE]
if not usb_resources:
    print("No USB instruments found. See troubleshooting below.")

In [ ]:
def open_instrument(address):
    inst = rm.open_resource(address)
    try:
        inst.timeout = TIMEOUT_MS
        inst.write_termination = "\n"
        inst.read_termination = "\n"
        return inst
    except Exception:
        inst.close()
        raise

identified = []
for address in usb_resources:
    try:
        with open_instrument(address) as inst:
            identity = inst.query("*IDN?").strip()
        print(address, "->", identity)
        fields = [field.strip() for field in identity.split(",")]
        identified.append((address, identity, fields))
    except (pyvisa.errors.VisaIOError, OSError, ValueError) as exc:
        print(address, "-> identification failed:", exc)

matches = [(address, identity) for address, identity, fields in identified
           if len(fields) >= 2 and "RIGOL" in fields[0].upper()
           and fields[1].upper().replace(" ", "").startswith("DG5502")]
print("Rigol DG5502 matches:", len(matches))

In [ ]:
# Keep the identified generator available for subsequent SCPI work.
if globals().get("generator") is not None:
    generator.close()
    generator = None
if len(matches) != 1:
    raise RuntimeError("Expected one DG5502. Check discovery output; set RESOURCE_OVERRIDE if multiple devices match.")
generator = open_instrument(matches[0][0])
try:
    identity = generator.query("*IDN?").strip()
    if identity != matches[0][1]:
        raise RuntimeError("Instrument identity changed: " + identity)
except Exception:
    generator.close()
    generator = None
    raise
print("Connected:", identity)

# For later SCPI work, use query() for commands returning a reply:
# print(generator.query("*IDN?"))
# Use generator.write("...") for commands from the DG5000 Pro programming manual.

## Troubleshooting
- Confirm the USB data cable uses the instrument's USB DEVICE port. Check Windows Device Manager for the instrument or an unknown USB device.
- If VISA cannot load, install/repair NI-VISA with USB support and restart the kernel. Check the Python path printed above.
- If no instrument appears, check NI MAX or your VISA vendor's connection utility. Close other instrument-control software, reconnect USB, and rerun discovery.
- If identification times out, close competing sessions, check remote-interface settings, and try increasing TIMEOUT_MS.
- Optional alternative: install `pyvisa-py pyusb` and set VISA_BACKEND to `"@py"`. USB also needs libusb and a compatible Windows USB driver. Python packages alone may be insufficient; changing the device driver may affect vendor software.
- Backend diagnostics: run `from pyvisa.util import get_debug_info; print(get_debug_info())` in a cell.

References: [PyVISA communication](https://pyvisa.readthedocs.io/en/latest/introduction/communication.html), [PyVISA-py USB prerequisites](https://pyvisa.readthedocs.io/projects/pyvisa-py/en/latest/installation.html).

In [ ]:
# Run when finished; closing sessions leaves output settings unchanged.
if globals().get("generator") is not None:
    generator.close()
    generator = None
if globals().get("rm") is not None:
    rm.close()
    rm = None
print("VISA sessions closed.")